In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import tifffile as tiff
import os
import time
import copy
import torchvision.transforms.v2 as T
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
DATA_DIR = '../data'
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
TARGETS = ['t_ch4', 't_no2', 't_co', 't_aai', 't_firms']
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
print(f"Using device: {DEVICE}")

In [ ]:
class ObscuraDataset(Dataset):
    def __init__(self, df, image_dir, feature_cols, target_cols=None, is_train=True, scaler=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.feature_cols = feature_cols
        self.target_cols = target_cols
        self.is_train = is_train
        self.image_col = 'sample_id'
        self.cache = {} 

        if is_train:
            self.scaler = StandardScaler()
            self.features = self.scaler.fit_transform(self.df[self.feature_cols].fillna(0))
            self.augment = T.Compose([
                T.RandomHorizontalFlip(p=0.5),
                T.RandomVerticalFlip(p=0.5),
                T.RandomRotation(degrees=90)
            ])
        else:
            self.scaler = scaler
            self.features = self.scaler.transform(self.df[self.feature_cols].fillna(0))
            self.augment = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if idx in self.cache:
            img = self.cache[idx].copy()
        else:
            img_name = f"{self.df.loc[idx, self.image_col]}.tif"
            img_path = os.path.join(self.image_dir, img_name)
            try:
                img = tiff.imread(img_path)
                if img.shape[0] != 14: img = np.transpose(img, (2, 0, 1))
                img = img.astype(np.float32)
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            except Exception:
                img = np.zeros((14, 384, 384), dtype=np.float32)
            self.cache[idx] = img.copy()

        img_tensor = torch.tensor(img)
        if self.augment is not None: img_tensor = self.augment(img_tensor)
        
        tabular = self.features[idx]

        if self.target_cols is not None:
            targets = self.df.loc[idx, self.target_cols].values.astype(np.float32)
            return img_tensor, torch.tensor(tabular, dtype=torch.float32), torch.tensor(targets)
        else:
            return img_tensor, torch.tensor(tabular, dtype=torch.float32)

In [ ]:
class ObscuraMultiTaskModel(nn.Module):
    def __init__(self, tabular_input_dim, num_targets=5):
        super(ObscuraMultiTaskModel, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(14, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)) 
        )
        self.mlp = nn.Sequential(
            nn.Linear(tabular_input_dim, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 32), nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Linear(128 + 32, 64), nn.ReLU(), nn.Linear(64, num_targets)
        )

    def forward(self, img, tabular):
        img_features = self.cnn(img).view(img.size(0), -1)
        tab_features = self.mlp(tabular)
        combined = torch.cat((img_features, tab_features), dim=1)
        return self.fc(combined)

class PhysicsInformedLoss(nn.Module):
    def __init__(self, physics_weight=0.01):
        super(PhysicsInformedLoss, self).__init__()
        self.physics_weight = physics_weight
        
    def forward(self, preds, targets):
        mask = ~torch.isnan(targets)
        targets_safe = torch.nan_to_num(targets, nan=0.0)
        mse_loss = (((preds - targets_safe) ** 2) * mask).sum() / (mask.sum() + 1e-8)
        
        # Physics Penalty: High fire > 1.0 but Smoke < 0.0
        fire_mask = (preds[:, 4] > 1.0).float()
        smoke_penalty = torch.relu(0.0 - preds[:, 3]) * fire_mask
        return mse_loss + (self.physics_weight * smoke_penalty.mean())

In [ ]:
train_df_full = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
features = [c for c in train_df_full.columns if c not in TARGETS and c not in ['sample_id', 'site_key', 'image_id']]

# Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train_df_full, groups=train_df_full['site_key']))
train = train_df_full.iloc[train_idx].reset_index(drop=True)
val = train_df_full.iloc[val_idx].reset_index(drop=True)

train_dataset = ObscuraDataset(train, IMAGE_DIR, features, TARGETS, is_train=True)
val_dataset = ObscuraDataset(val, IMAGE_DIR, features, TARGETS, is_train=False, scaler=train_dataset.scaler)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Training Loop
model = ObscuraMultiTaskModel(tabular_input_dim=len(features)).to(DEVICE)
criterion = PhysicsInformedLoss(physics_weight=0.01)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

EPOCHS = 50
best_val_loss = float('inf')
best_model_weights = None

print(f"Starting {EPOCHS}-Epoch training on {DEVICE}...")
for epoch in range(EPOCHS):
    start_time = time.time()
    
    model.train()
    train_loss = 0.0
    for images, tabular, targets_batch in train_loader:
        images, tabular, targets_batch = images.to(DEVICE), tabular.to(DEVICE), targets_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images, tabular), targets_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, tabular, targets_batch in val_loader:
            images, tabular, targets_batch = images.to(DEVICE), tabular.to(DEVICE), targets_batch.to(DEVICE)
            val_loss += criterion(model(images, tabular), targets_batch).item()
            
    avg_val = val_loss / len(val_loader)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_model_weights = copy.deepcopy(model.state_dict())
        saved_msg = "✅ Saved"
    else:
        saved_msg = ""
        
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f} | Time: {time.time()-start_time:.1f}s {saved_msg}")

# Save Model for Streamlit
model.load_state_dict(best_model_weights)
torch.save(model.state_dict(), '../obscura_model.pth')
print(f"\n✅ Training Complete! Best Val Loss: {best_val_loss:.4f}")

In [ ]:
gkf = GroupKFold(n_splits=5)
fold_predictions = []
best_val_losses = []

print("Starting 5-Fold Cross-Validation for Ensemble...")
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df_full, groups=train_df_full['site_key'])):
    train_f = train_df_full.iloc[train_idx].reset_index(drop=True)
    val_f = train_df_full.iloc[val_idx].reset_index(drop=True)
    
    train_ds = ObscuraDataset(train_f, IMAGE_DIR, features, TARGETS, is_train=True)
    val_ds = ObscuraDataset(val_f, IMAGE_DIR, features, TARGETS, is_train=False, scaler=train_ds.scaler)
    
    t_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    v_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    m = ObscuraMultiTaskModel(tabular_input_dim=len(features)).to(DEVICE)
    opt = optim.Adam(m.parameters(), lr=0.001, weight_decay=1e-4)
    crit = PhysicsInformedLoss(physics_weight=0.01)
    
    best_v_loss = float('inf')
    best_w = None
    epochs_no_improve = 0
    
    for epoch in range(15): # Use 15 epochs for speed
        m.train()
        for images, tabular, targets_batch in t_loader:
            images, tabular, targets_batch = images.to(DEVICE), tabular.to(DEVICE), targets_batch.to(DEVICE)
            opt.zero_grad()
            loss = crit(m(images, tabular), targets_batch)
            loss.backward()
            opt.step()
            
        m.eval()
        v_loss = 0.0
        with torch.no_grad():
            for images, tabular, targets_batch in v_loader:
                images, tabular, targets_batch = images.to(DEVICE), tabular.to(DEVICE), targets_batch.to(DEVICE)
                v_loss += crit(m(images, tabular), targets_batch).item()
        
        avg_v = v_loss / len(v_loader)
        if avg_v < best_v_loss:
            best_v_loss = avg_v
            best_w = copy.deepcopy(m.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= 4:
            break
            
    m.load_state_dict(best_w)
    best_val_losses.append(best_v_loss)
    print(f"  Fold {fold+1} Best Val Loss: {best_v_loss:.4f}")
    
    # Predict on Test
    test_ds = ObscuraDataset(test_df, IMAGE_DIR, features, is_train=False, scaler=train_ds.scaler)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    m.eval()
    fold_preds = []
    with torch.no_grad():
        for images, tabular in test_loader:
            images, tabular = images.to(DEVICE), tabular.to(DEVICE)
            fold_preds.append(m(images, tabular).cpu().numpy())
    fold_predictions.append(np.vstack(fold_preds))

print(f"\nAverage Validation Loss across folds: {np.mean(best_val_losses):.4f}")

final_test_preds = np.mean(fold_predictions, axis=0)
submission = pd.DataFrame({'sample_id': test_df['sample_id']})
for i, target in enumerate(TARGETS):
    submission[target] = final_test_preds[:, i]

submission.to_csv('../submission_pytorch_ensemble.csv', index=False)
print("✅ 5-Fold Ensemble Submission saved as submission_pytorch_ensemble.csv")